# GEC-RAG Project
## Notebook 03 — RAG Knowledge Base Construction

This notebook builds the **Pedagogical Knowledge Base (KB)** for the GEC-RAG system.

**Framework:** Corpus-driven, L2-general pedagogical feedback grounded in real learner error patterns from the Write & Improve Corpus 2024.

**Pipeline:**
1. Load ERRANT annotations from Notebook 02
2. Define priority error types based on corpus frequency
3. Extract representative examples per error type
4. Use GPT-4o to generate corpus-grounded pedagogical explanations
5. Build a structured KB mapping error types → examples → explanations
6. Index the KB using BM25 for fast retrieval
7. Test retrieval with sample queries

**Output:** An indexed, retrievable KB ready for the GEC-RAG pipeline in Notebook 06.

**Note on framing:** Explanations target L2 English learners generally. L1-specific
instantiation (e.g. L1-Mandarin) is identified as future work.

---
### Cell 1 — Imports & Setup

In [5]:
import pandas as pd
import numpy as np
import os
import json
import time
import pickle
import warnings
from collections import defaultdict
from openai import OpenAI
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
warnings.filterwarnings('ignore')

# ⚠️ Update BASE_DIR to match your actual path
BASE_DIR    = 'C:/gecrag'
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
KB_DIR      = os.path.join(BASE_DIR, 'knowledge_base')
os.makedirs(KB_DIR, exist_ok=True)

# Load API key
load_dotenv(os.path.join(BASE_DIR, '.env'))
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Quick API test
test = client.chat.completions.create(
    model='gpt-4o',
    messages=[{'role': 'user', 'content': 'Say OK'}],
    max_tokens=5
)
print(f'✅ OpenAI API connected: {test.choices[0].message.content}')
print(f'✅ All imports successful')

✅ OpenAI API connected: OK
✅ All imports successful


---
### Cell 2 — Load ERRANT Annotations

In [10]:
# Load specific (non-OTHER) annotations from Notebook 02
specific_path = os.path.join(RESULTS_DIR, 'errant_annotations_specific.csv')
summary_path  = os.path.join(RESULTS_DIR, 'errant_error_summary.csv')

df_specific = pd.read_csv(specific_path)
df_summary  = pd.read_csv(summary_path)

# Filter summary to specific only
df_summary_specific = df_summary[df_summary['is_specific'] == True].copy()

print(f'✅ Annotations loaded')
print(f'   Total specific error instances : {len(df_specific)}')
print(f'   Unique specific error types    : {df_specific["error_type"].nunique()}')
print(f'\nTop 15 error types by frequency:')
print(df_summary_specific.head(15)[['error_type','count','percentage']].to_string(index=False))

✅ Annotations loaded
   Total specific error instances : 50164
   Unique specific error types    : 48

Top 15 error types by frequency:
  error_type  count  percentage
      R:NOUN   8069       12.86
      R:ORTH   4089        6.52
     R:SPELL   3647        5.81
      R:PREP   3337        5.32
       M:DET   3273        5.22
       U:DET   2562        4.08
      R:VERB   2453        3.91
 R:VERB:FORM   2022        3.22
R:VERB:TENSE   1938        3.09
       R:DET   1903        3.03
  R:NOUN:NUM   1800        2.87
      M:PREP   1469        2.34
  R:VERB:SVA   1440        2.29
     R:MORPH   1270        2.02
      U:PREP    945        1.51


---
### Cell 3 — Define Priority Error Types
Prioritized purely by corpus frequency and pedagogical relevance to L2 English learners generally.
No L1-specific assumptions made at this stage.

In [4]:
# Tier 1: High frequency + high pedagogical value
# These are the errors most likely to benefit from explanation
TIER_1 = [
    'R:NOUN',         # Most frequent specific error — general noun errors
    'R:ORTH',         # Orthographic errors (capitalization, formatting)
    'M:DET',          # Missing determiner/article
    'R:SPELL',        # Spelling errors
    'R:NOUN:NUM',     # Singular/plural confusion
    'R:PREP',         # Wrong preposition
    'R:VERB:TENSE',   # Wrong verb tense
    'R:VERB',         # General verb errors
    'R:VERB:FORM',    # Wrong verb form
    'R:DET',          # Wrong determiner/article
]

# Tier 2: Medium frequency + pedagogically important
TIER_2 = [
    'R:MORPH',        # Morphological errors
    'U:DET',          # Unnecessary determiner/article
    'M:PREP',         # Missing preposition
    'R:VERB:SVA',     # Subject-verb agreement
    'U:PREP',         # Unnecessary preposition
    'M:VERB',         # Missing verb
    'R:PRON',         # Wrong pronoun
    'M:PRON',         # Missing pronoun
    'R:WO',           # Word order
    'R:ADJ',          # Adjective errors
]

ALL_PRIORITY = TIER_1 + TIER_2

# Check availability in corpus
available = df_specific['error_type'].unique()
print(f'Priority error types availability:')
print(f'\n{"Tier":<8} {"Error Type":<20} {"In Data":>10} {"Count":>10}')
print('-' * 52)
for t in TIER_1:
    count = len(df_specific[df_specific['error_type'] == t])
    status = '✅' if t in available else '❌'
    print(f'{"Tier 1":<8} {t:<20} {status:>10} {count:>10}')
print()
for t in TIER_2:
    count = len(df_specific[df_specific['error_type'] == t])
    status = '✅' if t in available else '❌'
    print(f'{"Tier 2":<8} {t:<20} {status:>10} {count:>10}')

Priority error types availability:

Tier     Error Type              In Data      Count
----------------------------------------------------
Tier 1   R:NOUN                        ✅       8069
Tier 1   R:ORTH                        ✅       4089
Tier 1   M:DET                         ✅       3273
Tier 1   R:SPELL                       ✅       3647
Tier 1   R:NOUN:NUM                    ✅       1800
Tier 1   R:PREP                        ✅       3337
Tier 1   R:VERB:TENSE                  ✅       1938
Tier 1   R:VERB                        ✅       2453
Tier 1   R:VERB:FORM                   ✅       2022
Tier 1   R:DET                         ✅       1903

Tier 2   R:MORPH                       ✅       1270
Tier 2   U:DET                         ✅       2562
Tier 2   M:PREP                        ✅       1469
Tier 2   R:VERB:SVA                    ✅       1440
Tier 2   U:PREP                        ✅        945
Tier 2   M:VERB                        ✅        919
Tier 2   R:PRON           

---
### Cell 4 — Extract Representative Examples per Error Type

In [5]:
def get_examples(df, error_type, n=5):
    """
    Get n representative examples for a given error type.
    Filters for clean, short, informative examples.
    """
    subset = df[df['error_type'] == error_type].copy()

    # Filter for clean examples
    subset = subset[
        (subset['original_span'].str.len() > 0) &
        (subset['correction'].str.len() > 0) &
        (subset['original_sent'].str.split().str.len() < 40) &
        (subset['original_sent'].str.split().str.len() > 5)
    ]

    n_sample = min(n, len(subset))
    sampled  = subset.sample(n=n_sample, random_state=42) if n_sample > 0 else subset

    examples = []
    for _, row in sampled.iterrows():
        examples.append({
            'original_sentence' : row['original_sent'],
            'corrected_sentence': row['corrected_sent'],
            'error_span'        : row['original_span'],
            'correction'        : row['correction'],
        })
    return examples


# Test with top error types
for test_type in ['M:DET', 'R:VERB:TENSE', 'R:NOUN:NUM']:
    examples = get_examples(df_specific, test_type, n=2)
    sep = '-'*55
    print(f'\n{sep}')
    print(f'Examples for {test_type}:')
    print(sep)
    for i, ex in enumerate(examples):
        print(f'  [{i+1}] Original : {ex["original_sentence"]}')
        print(f'       Corrected: {ex["corrected_sentence"]}')
        print(f'       Error    : "{ex["error_span"]}" → "{ex["correction"]}"')


-------------------------------------------------------
Examples for M:DET:
-------------------------------------------------------
  [1] Original : First people said that zoos are good to learn about animals I am with them but if the zookeeper doesn't punish animals or treat them like dirt.
       Corrected: The first people said that zoos are good to learn about animals and I am with them, but provided the zookeeper doesn't punish animals or treat them like dirt.
       Error    : "First" → "The first"
  [2] Original : In my opinion, Zoo is a good place to know about the life of wild animal as well as pet animals.
       Corrected: In my opinion, a zoo is a good place to learn about the life of wild animals as well as pet animals.
       Error    : "Zoo" → "a zoo"

-------------------------------------------------------
Examples for R:VERB:TENSE:
-------------------------------------------------------
  [1] Original : Unfortunately I couldn't be present to your party because my prof

---
### Cell 5 — GPT-4o Explanation Generator
Generates corpus-grounded, L2-general pedagogical explanations.
No L1-specific assumptions. Explanations are based on what the corpus data shows.

In [6]:
def generate_explanation(error_type, examples, frequency, total_errors, client):
    """
    Use GPT-4o to generate a corpus-grounded pedagogical explanation
    for an error type. Framed for general L2 English learners.
    """
    # Format examples
    examples_text = ''
    for i, ex in enumerate(examples[:5]):
        examples_text += f"""
  Example {i+1}:
    Original  : {ex['original_sentence']}
    Corrected : {ex['corrected_sentence']}
    Error span: "{ex['error_span']}" → "{ex['correction']}"
"""

    freq_pct = (frequency / total_errors * 100)

    prompt = f"""You are an expert EFL (English as a Foreign Language) teacher and 
corpus linguist. You are building a pedagogical knowledge base to help L2 English 
learners understand and correct their grammatical errors.

I am going to give you:
1. An error type code from the ERRANT toolkit: **{error_type}**
2. Real examples of this error drawn from a large corpus of L2 English learner writing
3. Frequency information: this error type accounts for {freq_pct:.1f}% of all errors 
   in the corpus, making it one of the most important to address

Real learner examples:
{examples_text}

Please write a clear, corpus-informed pedagogical explanation with this exact structure:

1. ERROR TYPE: A student-friendly name for this error (not the ERRANT code)

2. WHAT IT IS: A 1-2 sentence description of what this error involves, 
   based on the patterns you observe in the examples above

3. WHY IT IS DIFFICULT: Explain why this is a common challenge for L2 English 
   learners generally — what makes this aspect of English hard to acquire? 
   Base this on what the corpus examples suggest, not on any specific L1.

4. HOW TO FIX IT: A clear, actionable rule or strategy the learner can apply

5. CORPUS INSIGHT: One specific observation about the pattern of this error 
   as seen in real learner writing (based on the examples provided)

6. EXAMPLE: One clear before/after example with a brief explanation

Requirements:
- Write for a B1-B2 level English learner — clear and accessible
- Do not use linguistic jargon without explaining it
- Be encouraging and constructive in tone
- Ground observations in the actual examples provided, not generic textbook rules
- Do not make assumptions about the learner's native language"""

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=700,
        temperature=0.3
    )

    return response.choices[0].message.content


# Test with one error type before full run
total_errors = len(df_specific)
test_examples = get_examples(df_specific, 'M:DET', n=5)
test_freq     = len(df_specific[df_specific['error_type'] == 'M:DET'])

print('Testing GPT-4o explanation for M:DET...')
test_expl = generate_explanation('M:DET', test_examples, test_freq, total_errors, client)
sep = '='*60
print(f'\n{sep}')
print(test_expl)
print(sep)

Testing GPT-4o explanation for M:DET...

1. ERROR TYPE: Missing Articles

2. WHAT IT IS: This error occurs when a necessary article, such as "a," "an," or "the," is left out before a noun in a sentence. Articles help specify whether we are talking about something specific or general.

3. WHY IT IS DIFFICULT: Many languages use articles differently or not at all, which can make it challenging for English learners to know when and which article to use. In English, articles are important for clarity and meaning, and their correct use can be tricky to master.

4. HOW TO FIX IT: When you write, ask yourself if the noun you are using is specific or general. Use "the" for specific nouns that both you and your reader know about, and "a" or "an" for general nouns that are new or not specifically identified.

5. CORPUS INSIGHT: In the examples provided, learners often forget to use "the" when referring to something specific, like "the first people" or "the rest of the students," and "a" when int

---
### Cell 6 — Build Full KB
> ⏳ Calls GPT-4o for each priority error type. ~3-5 minutes. ~$0.50-1.00 in API credits.

In [7]:
knowledge_base = {}
failed         = []
total_errors   = len(df_specific)

print(f'Building KB for {len(ALL_PRIORITY)} error types...')
print(f'{"Error Type":<20} {"Freq":>8} {"Examples":>10} {"Status":>10}')
print('-' * 52)

for error_type in ALL_PRIORITY:
    if error_type not in df_specific['error_type'].unique():
        print(f'{error_type:<20} {"N/A":>8} {"N/A":>10} {"SKIPPED":>10}')
        continue

    try:
        freq     = len(df_specific[df_specific['error_type'] == error_type])
        examples = get_examples(df_specific, error_type, n=5)

        if not examples:
            print(f'{error_type:<20} {freq:>8} {0:>10} {"NO EXAMPLES":>10}')
            continue

        explanation = generate_explanation(
            error_type, examples, freq, total_errors, client
        )

        knowledge_base[error_type] = {
            'error_type'       : error_type,
            'tier'             : 1 if error_type in TIER_1 else 2,
            'frequency'        : freq,
            'frequency_pct'    : round(freq / total_errors * 100, 2),
            'examples'         : examples,
            'explanation'      : explanation,
            'manually_reviewed': False,
            'retrieval_keys'   : [
                error_type,
                error_type.replace(':', ' ').lower(),
                error_type.split(':')[-1].lower() if ':' in error_type
                else error_type.lower()
            ]
        }

        print(f'{error_type:<20} {freq:>8} {len(examples):>10} {"✅ OK":>10}')
        time.sleep(0.5)  # Rate limiting

    except Exception as e:
        failed.append(error_type)
        print(f'{error_type:<20} {"ERROR":>8} {"-":>10} {str(e)[:15]:>10}')

print(f'\n✅ KB built: {len(knowledge_base)} entries')
if failed:
    print(f'❌ Failed  : {failed}')

Building KB for 20 error types...
Error Type               Freq   Examples     Status
----------------------------------------------------
R:NOUN                   8069          5       ✅ OK
R:ORTH                   4089          5       ✅ OK
M:DET                    3273          5       ✅ OK
R:SPELL                  3647          5       ✅ OK
R:NOUN:NUM               1800          5       ✅ OK
R:PREP                   3337          5       ✅ OK
R:VERB:TENSE             1938          5       ✅ OK
R:VERB                   2453          5       ✅ OK
R:VERB:FORM              2022          5       ✅ OK
R:DET                    1903          5       ✅ OK
R:MORPH                  1270          5       ✅ OK
U:DET                    2562          5       ✅ OK
M:PREP                   1469          5       ✅ OK
R:VERB:SVA               1440          5       ✅ OK
U:PREP                    945          5       ✅ OK
M:VERB                    919          5       ✅ OK
R:PRON                    889

---
### Cell 7 — Review Generated Explanations
> **Human review step.** Read through all explanations carefully.
> Check for: accuracy, clarity, tone, corpus grounding.
> Note any error types that need editing.

In [8]:
print('=== KB EXPLANATION REVIEW ===')
print('Read carefully. Note any entries needing manual editing.\n')
print('Key things to check per entry:')
print('  - Is the student-friendly name accurate?')
print('  - Is the explanation grounded in the examples shown?')
print('  - Is the fix actionable and clear?')
print('  - Is the tone appropriate for a B1-B2 learner?')

sep1 = '='*60
sep2 = '-'*60

for error_type, entry in knowledge_base.items():
    tier = entry['tier']
    freq = entry['frequency']
    pct  = entry['frequency_pct']
    print(f'\n{sep1}')
    print(f'[TIER {tier}] {error_type} | Frequency: {freq} ({pct}% of errors)')
    print(sep2)
    print(f'Corpus examples used:')
    for i, ex in enumerate(entry['examples'][:2]):
        print(f'  [{i+1}] "{ex["error_span"]}" → "{ex["correction"]}"')
    print(sep2)
    print(entry['explanation'])
    print()

=== KB EXPLANATION REVIEW ===
Read carefully. Note any entries needing manual editing.

Key things to check per entry:
  - Is the student-friendly name accurate?
  - Is the explanation grounded in the examples shown?
  - Is the fix actionable and clear?
  - Is the tone appropriate for a B1-B2 learner?

[TIER 1] R:NOUN | Frequency: 8069 (16.09% of errors)
------------------------------------------------------------
Corpus examples used:
  [1] "friend," → "friend."
  [2] "educations" → "education,"
------------------------------------------------------------
1. ERROR TYPE: Incorrect Noun Usage

2. WHAT IT IS: This error involves using nouns incorrectly, often by adding unnecessary endings or using the wrong form of the noun in a sentence.

3. WHY IT IS DIFFICULT: English nouns can be tricky because they have different forms depending on whether they are singular or plural, and some nouns don't follow regular patterns. Additionally, punctuation around nouns can be confusing, as it changes

---
### Cell 8 — Manual Edit Interface
> Use this cell to edit any explanation that needs improvement after Cell 7 review.
> Prioritize Tier 1 entries.

In [2]:
import json
import os

BASE_DIR = 'C:/gecrag'  # ⚠️ update if your path is different
KB_DIR   = os.path.join(BASE_DIR, 'knowledge_base')

# Reload KB from disk
with open(os.path.join(KB_DIR, 'knowledge_base.json'), 'r', encoding='utf-8') as f:
    knowledge_base = json.load(f)

print(f'✅ KB reloaded: {len(knowledge_base)} entries')

# Redefine edit function
def edit_explanation(knowledge_base, error_type, new_explanation):
    """Replace an explanation with a manually edited version."""
    if error_type in knowledge_base:
        knowledge_base[error_type]['explanation']       = new_explanation
        knowledge_base[error_type]['manually_reviewed'] = True
        print(f'✅ Updated: {error_type}')
    else:
        print(f'❌ Not found: {error_type}')
    return knowledge_base

✅ KB reloaded: 20 entries


In [3]:
def edit_explanation(knowledge_base, error_type, new_explanation):
    """Replace an explanation with a manually edited version."""
    if error_type in knowledge_base:
        knowledge_base[error_type]['explanation']       = new_explanation
        knowledge_base[error_type]['manually_reviewed'] = True
        print(f'✅ Updated: {error_type}')
    else:
        print(f'❌ Not found: {error_type}')
    return knowledge_base


def regenerate_explanation(knowledge_base, error_type, df_specific,
                            total_errors, client, additional_instruction=''):
    """
    Regenerate explanation for a specific error type with optional
    additional instruction to guide GPT-4o.
    """
    if error_type not in knowledge_base:
        print(f'❌ Not found: {error_type}')
        return knowledge_base

    examples = get_examples(df_specific, error_type, n=5)
    freq     = knowledge_base[error_type]['frequency']

    # Add any additional instruction to the prompt
    original_fn = generate_explanation
    new_expl = generate_explanation(
        error_type + (f' ({additional_instruction})' if additional_instruction else ''),
        examples, freq, total_errors, client
    )
    knowledge_base[error_type]['explanation']       = new_expl
    knowledge_base[error_type]['manually_reviewed'] = True
    print(f'✅ Regenerated: {error_type}')
    print(new_expl)
    return knowledge_base


# ============================================================
# MANUAL REPLACEMENTS FOR THREE FLAGGED ENTRIES
# ============================================================

# --- R:NOUN ---
# Problem: conflated noun form errors with punctuation errors
knowledge_base = edit_explanation(
    knowledge_base,
    'R:NOUN',
    """1. ERROR TYPE: Wrong Noun Form

2. WHAT IT IS: This error involves using the wrong form of a noun — for example, using a countable form when an uncountable one is needed, choosing the wrong word class (a verb instead of a noun), or replacing one noun with another that does not fit the meaning of the sentence.

3. WHY IT IS DIFFICULT: English nouns are challenging because some have irregular forms, some cannot be made plural at all (uncountable nouns like "education" or "information"), and many words look similar but belong to different word classes. Learners often apply regular rules (adding -s, -tion, -ment) where they do not apply.

4. HOW TO FIX IT: When you write a noun, ask yourself two questions: (1) Is this noun countable or uncountable in this context? Uncountable nouns like "education," "advice," and "information" cannot take a plural -s. (2) Is this the right word class? If you need a noun, check that you are not using a verb or adjective form instead.

5. CORPUS INSIGHT: A very common pattern in learner writing is adding -s to uncountable nouns — for example, writing "educations" instead of "education," or "informations" instead of "information." These nouns refer to general concepts and do not have a plural form in English.

6. EXAMPLE:
   - Before: "Schools need to keep their levels of educations high."
   - After: "Schools need to keep their levels of education high."
   - Explanation: "Education" is an uncountable noun in English and cannot be made plural by adding -s. It refers to the general concept, not individual units."""
)

# --- M:PREP ---
# Problem: corpus insight was oddly specific about "The" and "One"
# which reflects the examples but misleads about the general error
knowledge_base = edit_explanation(
    knowledge_base,
    'M:PREP',
    """1. ERROR TYPE: Missing Preposition

2. WHAT IT IS: This error occurs when a necessary preposition is left out of a sentence or phrase. Prepositions are small words like "in," "on," "with," "for," and "like" that show the relationship between ideas — location, time, manner, or comparison.

3. WHY IT IS DIFFICULT: Prepositions in English do not translate directly from other languages. The choice of preposition often depends on fixed expressions and context rather than logical rules, making them hard to learn systematically. Learners frequently omit them when the sentence still feels readable without them.

4. HOW TO FIX IT: When you begin a sentence or phrase, ask yourself what relationship you are trying to express: location ("In the city..."), manner ("With hard work..."), comparison ("Like many students..."), or time ("On Monday..."). If the sentence describes a setting or context, it almost certainly needs an opening preposition.

5. CORPUS INSIGHT: A common pattern in learner writing is beginning a sentence directly with a noun phrase — for example "The world being interconnected..." or "One hand..." — when the sentence actually requires an introductory preposition phrase such as "With the world being interconnected..." or "On one hand..." The noun phrase is correct but the preposition that should introduce it is missing.

6. EXAMPLE:
   - Before: "The world being so interconnected, you must be able to interact with people from other cultures."
   - After: "With the world being so interconnected, you must be able to interact with people from other cultures."
   - Explanation: The preposition "With" is needed to introduce the context or condition being described. Without it, the opening phrase is grammatically incomplete."""
)

# --- R:SPELL ---
# Problem: fix advice was generic — not grounded in corpus patterns
knowledge_base = edit_explanation(
    knowledge_base,
    'R:SPELL',
    """1. ERROR TYPE: Spelling Error

2. WHAT IT IS: This error occurs when a word is spelled incorrectly. The misspelled word may still look or sound like the intended word, but the incorrect spelling can change the meaning of the sentence or make it harder to understand.

3. WHY IT IS DIFFICULT: English spelling is not fully phonetic — many words are not spelled the way they sound. This is especially difficult for learners because similar-sounding words can have very different spellings (for example, "than" vs "that," or "worm" vs "warm"). Errors often happen not from carelessness but from relying on how a word sounds rather than how it is spelled.

4. HOW TO FIX IT: After writing, re-read your text slowly and focus on individual words rather than meaning. Pay special attention to words you are less familiar with. For words that look unusual, double-check by breaking them into parts — for example, "psy-cho-lo-gi-cal" — to identify where the error may be. A spell-checker can help catch obvious errors but will miss cases where a misspelling produces a real word (e.g. "worm" instead of "warm").

5. CORPUS INSIGHT: A notable pattern in learner writing is spelling errors that produce a different real English word, making them easy to miss. For example, writing "worm" instead of "warm," or "than" instead of "that." These are particularly problematic because spell-checkers will not flag them — careful re-reading is the only reliable way to catch them.

6. EXAMPLE:
   - Before: "I am pleased with the comfortable environment, like worm and cozy rooms."
   - After: "I am pleased with the comfortable environment, like warm and cozy rooms."
   - Explanation: "Worm" is a real English word but wrong here — the intended word is "warm." This type of spelling error cannot be caught automatically and requires careful reading."""
)

# Confirm
print('Updated entries:')
for et in ['R:NOUN', 'M:PREP', 'R:SPELL']:
    reviewed = '✅' if knowledge_base[et]['manually_reviewed'] else '❌'
    print(f'  {reviewed} {et}')

reviewed = sum(1 for e in knowledge_base.values() if e['manually_reviewed'])
print(f'Entries manually reviewed: {reviewed}/{len(knowledge_base)}')
print('\n💡 Tip: Focus on Tier 1 entries first — these appear most in the pipeline.')

✅ Updated: R:NOUN
✅ Updated: M:PREP
✅ Updated: R:SPELL
Updated entries:
  ✅ R:NOUN
  ✅ M:PREP
  ✅ R:SPELL
Entries manually reviewed: 3/20

💡 Tip: Focus on Tier 1 entries first — these appear most in the pipeline.


---
### Cell 9 — Build BM25 Retrieval Index

In [6]:
def build_bm25_index(knowledge_base):
    """
    Build a BM25 index over KB entries.
    Each document combines error type code, retrieval keys,
    explanation text, and example sentences for rich matching.
    """
    documents    = []
    error_types  = []

    for error_type, entry in knowledge_base.items():
        # Build rich document for indexing
        example_text = ' '.join([
            f"{ex['original_sentence']} {ex['corrected_sentence']}"
            for ex in entry['examples'][:3]
        ])

        doc = (
            f"{error_type} "
            f"{' '.join(entry['retrieval_keys'])} "
            f"{entry['explanation']} "
            f"{example_text}"
        )

        tokens = doc.lower().split()
        documents.append(tokens)
        error_types.append(error_type)

    bm25 = BM25Okapi(documents)
    return bm25, error_types, documents


bm25_index, index_error_types, index_documents = build_bm25_index(knowledge_base)

print(f'✅ BM25 index built')
print(f'   Documents indexed : {len(index_documents)}')
print(f'   Error types       : {index_error_types}')

✅ BM25 index built
   Documents indexed : 20
   Error types       : ['R:NOUN', 'R:ORTH', 'M:DET', 'R:SPELL', 'R:NOUN:NUM', 'R:PREP', 'R:VERB:TENSE', 'R:VERB', 'R:VERB:FORM', 'R:DET', 'R:MORPH', 'U:DET', 'M:PREP', 'R:VERB:SVA', 'U:PREP', 'M:VERB', 'R:PRON', 'M:PRON', 'R:WO', 'R:ADJ']


---
### Cell 10 — Test Retrieval

In [7]:
def retrieve(query, bm25, error_types, knowledge_base, top_k=3):
    """
    Retrieve top-k KB entries for a given query.
    Accepts either ERRANT error type codes or natural language.
    """
    tokens  = query.lower().split()
    scores  = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        if scores[idx] > 0:
            et = error_types[idx]
            results.append({
                'error_type' : et,
                'score'      : round(float(scores[idx]), 3),
                'tier'       : knowledge_base[et]['tier'],
                'frequency'  : knowledge_base[et]['frequency'],
                'explanation': knowledge_base[et]['explanation'],
                'examples'   : knowledge_base[et]['examples'][:2],
            })
    return results


# Test queries — mix of error codes and natural language
test_queries = [
    'R:VERB:TENSE',
    'M:DET',
    'R:NOUN:NUM',
    'missing article',
    'wrong verb tense',
    'singular plural noun',
    'preposition error',
    'subject verb agreement',
    'spelling mistake',
    'word order',
]

sep = '-'*55
print('Retrieval Test Results:')
print(sep)
for query in test_queries:
    results = retrieve(query, bm25_index, index_error_types, knowledge_base, top_k=1)
    if results:
        r = results[0]
        print(f'Query : "{query}"')
        print(f'Match : {r["error_type"]} (score: {r["score"]}, tier: {r["tier"]})')
        print(f'Preview: {r["explanation"][:120]}...')
    else:
        print(f'Query : "{query}" → No match found')
    print(sep)

Retrieval Test Results:
-------------------------------------------------------
Query : "R:VERB:TENSE"
Match : R:VERB:TENSE (score: 3.614, tier: 1)
Preview: 1. ERROR TYPE: Verb Tense Confusion

2. WHAT IT IS: This error involves using the wrong verb tense in a sentence, which ...
-------------------------------------------------------
Query : "M:DET"
Match : M:DET (score: 3.788, tier: 1)
Preview: 1. ERROR TYPE: Missing Articles

2. WHAT IT IS: This error involves leaving out necessary articles like "a," "an," or "t...
-------------------------------------------------------
Query : "R:NOUN:NUM"
Match : R:NOUN:NUM (score: 3.564, tier: 1)
Preview: 1. ERROR TYPE: Singular and Plural Noun Confusion

2. WHAT IT IS: This error occurs when learners mistakenly use a plura...
-------------------------------------------------------
Query : "missing article"
Match : R:DET (score: 4.002, tier: 1)
Preview: 1. ERROR TYPE: Misuse of Articles and Demonstratives

2. WHAT IT IS: This error involves using

---
### Cell 11 — Save KB & Index

In [8]:
# Save KB as JSON (human readable)
kb_json_path = os.path.join(KB_DIR, 'knowledge_base.json')
with open(kb_json_path, 'w', encoding='utf-8') as f:
    json.dump(knowledge_base, f, ensure_ascii=False, indent=2)
print(f'✅ KB saved          : {kb_json_path}')

# Save BM25 index
index_path = os.path.join(KB_DIR, 'bm25_index.pkl')
with open(index_path, 'wb') as f:
    pickle.dump({
        'bm25'       : bm25_index,
        'error_types': index_error_types,
        'documents'  : index_documents
    }, f)
print(f'✅ BM25 index saved  : {index_path}')

# Save flat CSV summary
kb_rows = []
for et, entry in knowledge_base.items():
    kb_rows.append({
        'error_type'       : et,
        'tier'             : entry['tier'],
        'frequency'        : entry['frequency'],
        'frequency_pct'    : entry['frequency_pct'],
        'n_examples'       : len(entry['examples']),
        'manually_reviewed': entry['manually_reviewed'],
        'explanation_len'  : len(entry['explanation']),
    })
kb_df = pd.DataFrame(kb_rows)
kb_csv_path = os.path.join(KB_DIR, 'knowledge_base_summary.csv')
kb_df.to_csv(kb_csv_path, index=False)
print(f'✅ KB summary saved  : {kb_csv_path}')

✅ KB saved          : C:/gecrag\knowledge_base\knowledge_base.json
✅ BM25 index saved  : C:/gecrag\knowledge_base\bm25_index.pkl
✅ KB summary saved  : C:/gecrag\knowledge_base\knowledge_base_summary.csv


---
### Cell 12 — Summary Report

In [11]:
sep1 = '='*60
sep2 = '-'*45

print(sep1)
print('RAG KNOWLEDGE BASE — FINAL SUMMARY')
print(sep1)

tier1_entries = {k:v for k,v in knowledge_base.items() if v['tier']==1}
tier2_entries = {k:v for k,v in knowledge_base.items() if v['tier']==2}

print(f'\n📊 KB Statistics:')
print(f'   Total entries         : {len(knowledge_base)}')
print(f'   Tier 1 (high priority): {len(tier1_entries)}')
print(f'   Tier 2 (medium)       : {len(tier2_entries)}')
print(f'   Manually reviewed     : {sum(1 for e in knowledge_base.values() if e["manually_reviewed"])}')

covered = df_specific[df_specific['error_type'].isin(knowledge_base.keys())]
pct     = len(covered) / len(df_specific) * 100
print(f'\n📊 Corpus Coverage:')
print(f'   Errors covered by KB  : {len(covered):,}/{len(df_specific):,} ({pct:.1f}%)')

print(f'\n📊 Tier 1 Entries:')
print(f'   {"Error Type":<20} {"Freq":>8} {"% Total":>8} {"Reviewed":>10}')
print(f'   {sep2}')
for et, entry in tier1_entries.items():
    reviewed = '✅' if entry['manually_reviewed'] else '⚠️  pending'
    print(f'   {et:<20} {entry["frequency"]:>8} '
          f'{entry["frequency_pct"]:>7.1f}% {reviewed:>10}')

print(f'\n📁 Output Files:')
print(f'   - knowledge_base/knowledge_base.json       (full KB)')
print(f'   - knowledge_base/bm25_index.pkl            (retrieval index)')
print(f'   - knowledge_base/knowledge_base_summary.csv')

print(f'\n⚠️  Recommended Action Before Proceeding:')
print(f'   1. Open knowledge_base/knowledge_base.json')
print(f'   2. Read through all Tier 1 explanations')
print(f'   3. Edit any inaccurate entries using Cell 8')
print(f'   4. Re-run Cell 11 to save updated KB')

print(f'\n➡️  Next: Run Notebook 04 — GECToR Baseline')

RAG KNOWLEDGE BASE — FINAL SUMMARY

📊 KB Statistics:
   Total entries         : 20
   Tier 1 (high priority): 10
   Tier 2 (medium)       : 10
   Manually reviewed     : 3

📊 Corpus Coverage:
   Errors covered by KB  : 44,131/50,164 (88.0%)

📊 Tier 1 Entries:
   Error Type               Freq  % Total   Reviewed
   ---------------------------------------------
   R:NOUN                   8069    16.1%          ✅
   R:ORTH                   4089     8.2% ⚠️  pending
   M:DET                    3273     6.5% ⚠️  pending
   R:SPELL                  3647     7.3%          ✅
   R:NOUN:NUM               1800     3.6% ⚠️  pending
   R:PREP                   3337     6.7% ⚠️  pending
   R:VERB:TENSE             1938     3.9% ⚠️  pending
   R:VERB                   2453     4.9% ⚠️  pending
   R:VERB:FORM              2022     4.0% ⚠️  pending
   R:DET                    1903     3.8% ⚠️  pending

📁 Output Files:
   - knowledge_base/knowledge_base.json       (full KB)
   - knowledge_base/bm25_in